# Step 9: Tuning a CNN
A LeNet-style 1D-CNN on the 25 frames of each second. Grid search: 2 filter counts x 2 kernel sizes x 2 dropouts = 8 settings on a validation set; the winner is retrained and tested once (development stage, random split).
*Same results as `src/09_cnn.py`.*

**In short** (what this notebook shows)

- **What:** CNN tuning: 8 settings of number of filters, kernel size and dropout on the 25 frames of each second.
- **Needs the NoXi data?** Yes. Random split (development step).
- **Result / use:** Best: 64 filters, kernel 5, dropout 0.1 (test 0.791, unstable). Shows how convolutions work on a sequence.

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: **24 conversations** (English 026, 027, 028, 029, 030, 058, 062, 063; French 001, 007, 009, 011, 013, 014, 020, 021; German 068, 070, 072, 073, 074, 075, 076, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

**Imports.**

In [3]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
keras.utils.set_random_seed(42)

In [4]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Load the 25-frame sequences, split 60 / 20 / 20, and scale every frame** with a scaler fitted on training frames only.

In [5]:
y = pd.read_parquet("data/features/noxi_engagement.parquet", columns=["engagement"])["engagement"].to_numpy("float32")
X = np.load("data/features/noxi_sequences.npy", mmap_mode="r")   # read from disk piece by piece        # windows x 25 frames x 802
rest, test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
train, val = train_test_split(rest, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X[train].reshape(-1, 802))


def prepare(rows):
    return scaler.transform(X[rows].reshape(-1, 802)).reshape(len(rows), 25, 802).astype("float32")


X_train, X_val, X_test = prepare(train), prepare(val), prepare(test)
y_train, y_val, y_test = y[train], y[val], y[test]
del X
print("train", len(X_train), "| val", len(X_val), "| test", len(X_test))

train 29091 | val 9697 | test 9698


**The CNN (LeNet style):** two blocks of convolution + max pooling, then two dense layers.

In [6]:
def build(filters, kernel, dropout):
    return keras.Sequential([
        layers.Input(shape=(25, 802)),
        layers.Conv1D(filters, kernel, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(filters * 2, kernel, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(1),
    ])


def run(filters, kernel, dropout, epochs=15):
    keras.utils.set_random_seed(42)        # every setting starts the same
    model = build(filters, kernel, dropout)
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    model.fit(X_train, y_train, epochs=epochs, batch_size=128, verbose=0)
    return ccc(y_val, model.predict(X_val, verbose=0).flatten()), model

**Grid search: 8 settings.**

In [7]:
rows = []
for filters in [32, 64]:
    for kernel in [3, 5]:
        for dropout in [0.1, 0.3]:
            val_ccc, _ = run(filters, kernel, dropout)
            rows.append({"filters": filters, "kernel": kernel, "dropout": dropout, "val_ccc": val_ccc})
            print(f"filters {filters}  kernel {kernel}  dropout {dropout}  val CCC {val_ccc:.3f}")
results = pd.DataFrame(rows).sort_values("val_ccc", ascending=False).reset_index(drop=True)
print(results.round(4).to_string(index=False))

filters 32  kernel 3  dropout 0.1  val CCC 0.533


filters 32  kernel 3  dropout 0.3  val CCC 0.312


filters 32  kernel 5  dropout 0.1  val CCC 0.649


filters 32  kernel 5  dropout 0.3  val CCC 0.755


filters 64  kernel 3  dropout 0.1  val CCC 0.818


filters 64  kernel 3  dropout 0.3  val CCC 0.726


filters 64  kernel 5  dropout 0.1  val CCC 0.821


filters 64  kernel 5  dropout 0.3  val CCC 0.566
 filters  kernel  dropout  val_ccc
      64       5      0.1   0.8208
      64       3      0.1   0.8181
      32       5      0.3   0.7546
      64       3      0.3   0.7257
      32       5      0.1   0.6486
      64       5      0.3   0.5663
      32       3      0.1   0.5329
      32       3      0.3   0.3116


**Retrain the winner for 30 epochs and use the test set once.**

In [8]:
w = results.iloc[0]
val_ccc, model = run(int(w["filters"]), int(w["kernel"]), float(w["dropout"]), epochs=30)
pred = model.predict(X_test, verbose=0).flatten()
print(f"best: {int(w['filters'])} -> {int(w['filters']) * 2} filters, kernel {int(w['kernel'])}, dropout {w['dropout']}")
print(f"validation CCC {val_ccc:.3f} | TEST CCC {ccc(y_test, pred):.3f} | Pearson {np.corrcoef(y_test, pred)[0, 1]:.3f} "
      f"| MAE {mean_absolute_error(y_test, pred):.3f}")
model.summary()

best: 64 -> 128 filters, kernel 5, dropout 0.1
validation CCC 0.733 | TEST CCC 0.791 | Pearson 0.796 | MAE 0.067


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_16 (Conv1D)                   │ (None, 21, 64)              │         256,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_16 (MaxPooling1D)      │ (None, 10, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_17 (Conv1D)                   │ (None, 6, 128)              │          41,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_17 (MaxPooling1D)      │ (None, 3, 128)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_8 (Flatten)                  │ (None, 384)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_24 (Dense)                     │ (None, 128)                 │          49,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_16 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_25 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_17 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_26 (Dense)                     │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,066,181 (4.07 MB)

 Trainable params: 355,393 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 710,788 (2.71 MB)

**Save the results** (`reports/cnn_tuning_results.csv`).

In [9]:
results.to_csv("reports/cnn_tuning_results.csv", index=False)
print("saved reports/cnn_tuning_results.csv")

saved reports/cnn_tuning_results.csv
